<a href="https://colab.research.google.com/github/peremartra/fairness-pruning/blob/main/notebooks/03_neuron_bias_detection_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Neuron Bias Detection (English)

**Model:** `meta-llama/Llama-3.2-1B`  
**Dataset:** `oopere/fairness-pruning-pairs-en`  
**Categories:** `PhysicalAppearance` (15 pairs), `Gender` (15 pairs)  
**Library:** OptiPFair — `analyze_neuron_bias` + `compute_fairness_pruning_scores`

### Workflow
1. For each category: compute per-neuron bias scores from prompt pairs
2. Combine bias + structural importance → fairness pruning scores
3. Cross-category analysis: identify shared top candidates

### Outputs
```
fairness-pruning/results/llama-3.2-1B/bias_detection/en/
├── PhysicalAppearance/
│   ├── bias_scores.pt / bias_scores.json
│   ├── fairness_scores.pt / fairness_scores.json
├── Gender/
│   ├── bias_scores.pt / bias_scores.json
│   ├── fairness_scores.pt / fairness_scores.json
└── comparison_summary.json
```

## Cell 1 — Install dependencies

In [1]:
!pip install -q "optipfair[viz]" datasets transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 5.7 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch

## Cell 2 — Configuration

Edit these values to change the experiment parameters.

In [3]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = "meta-llama/Llama-3.2-1B"

# ── Dataset ────────────────────────────────────────────────────────────────
DATASET_NAME   = "oopere/fairness-pruning-pairs-en"
CATEGORIES     = ["PhysicalAppearance", "Gender"]

# ── OptiPFair — analyze_neuron_bias ────────────────────────────────────────
TARGET_LAYERS  = ["gate_proj", "up_proj"]   # GLU MLP projections to analyze
AGGREGATION    = "mean"                      # "mean" | "max" across sequence
BATCH_SIZE     = 4                           # Prompt pairs per batch (reduce if OOM)

# ── OptiPFair — compute_fairness_pruning_scores ────────────────────────────
# 0.0 = importance-only (standard pruning)  |  1.0 = bias-only
# 0.45 = balanced, slight edge toward structural importance
BIAS_WEIGHT    = 0.45

# ── Comparative analysis ───────────────────────────────────────────────────
# Percentile threshold to select top neuron candidates per layer
TOP_PERCENT    = 5.0

# ── Google Drive paths ─────────────────────────────────────────────────────
DRIVE_BASE     = "/content/drive/MyDrive/fairness-pruning"
RESULTS_BASE   = f"{DRIVE_BASE}/results/llama-3.2-1B/bias_detection/en"

In [4]:
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

## Cell 3 — GPU detection and dtype assignment

In [5]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_capability = torch.cuda.get_device_capability(0)

    # bfloat16 is natively supported from compute capability 8.0 onwards
    # (Ampere, Ada Lovelace, Hopper...). Below 8.0 (e.g. T4 = 7.5) use float16.
    if compute_capability[0] >= 8:
        DTYPE = torch.bfloat16
    else:
        DTYPE = torch.float16

    print(f"✅ GPU detected: {gpu_name}")
    print(f"   Compute capability: {compute_capability[0]}.{compute_capability[1]}")
    print(f"   VRAM: {vram_gb:.1f} GB")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("⚠️  No GPU detected — running on CPU with float32 (slow)")

print(f"   dtype → {DTYPE}")

✅ GPU detected: NVIDIA L4
   Compute capability: 8.9
   VRAM: 23.7 GB
   dtype → torch.bfloat16


## Cell 4 — Mount Google Drive and create output directories

In [6]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Create output directories for each category + the shared en/ root
dirs_to_create = [RESULTS_BASE] + [f"{RESULTS_BASE}/{cat}" for cat in CATEGORIES]

for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)
    print(f"📁 {d}")

print("\n✅ Output directories ready.")

Mounted at /content/drive
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/PhysicalAppearance
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/Gender

✅ Output directories ready.


## Cell 5 — Load model and tokenizer

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME} (dtype={DTYPE})")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

print(f"\n✅ Model loaded.")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"   Layers    : {model.config.num_hidden_layers}")

Loading tokenizer: meta-llama/Llama-3.2-1B


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]


✅ Model loaded.
   Parameters: 1.24B
   Layers    : 16


## Helper — serialization utilities

Both `bias_scores` (`Dict[str, Tensor]`) and `fairness_scores` (`Dict[int, Tensor]`)
are saved in two formats:
- `.pt` — native PyTorch, for downstream OptiPFair calls
- `.json` — human-readable, with `shape` stored separately from `values`

In [8]:
import json


def save_bias_scores(bias_scores: dict, out_dir: str) -> None:
    """Save bias_scores (Dict[str, Tensor]) as .pt and .json."""
    pt_path   = os.path.join(out_dir, "bias_scores.pt")
    json_path = os.path.join(out_dir, "bias_scores.json")

    # .pt — full tensors for OptiPFair
    torch.save(bias_scores, pt_path)

    # .json — shape + values for inspection
    serialized = {
        layer_name: {
            "shape" : list(tensor.shape),
            "values": tensor.cpu().float().tolist(),
        }
        for layer_name, tensor in bias_scores.items()
    }
    with open(json_path, "w") as f:
        json.dump(serialized, f, indent=2)

    print(f"   💾 bias_scores → {pt_path}")
    print(f"   💾 bias_scores → {json_path}")


def save_fairness_scores(fairness_scores: dict, out_dir: str) -> None:
    """Save fairness_scores (Dict[int, Tensor]) as .pt and .json."""
    pt_path   = os.path.join(out_dir, "fairness_scores.pt")
    json_path = os.path.join(out_dir, "fairness_scores.json")

    # .pt — full tensors
    torch.save(fairness_scores, pt_path)

    # .json — string keys (JSON requires string keys), shape + values
    serialized = {
        str(layer_idx): {
            "shape" : list(tensor.shape),
            "values": tensor.cpu().float().tolist(),
        }
        for layer_idx, tensor in fairness_scores.items()
    }
    with open(json_path, "w") as f:
        json.dump(serialized, f, indent=2)

    print(f"   💾 fairness_scores → {pt_path}")
    print(f"   💾 fairness_scores → {json_path}")


def load_prompt_pairs(dataset_name: str, category: str) -> list:
    """Load (prompt_1, prompt_2) tuples from a HuggingFace dataset subset."""
    from datasets import load_dataset
    ds = load_dataset(dataset_name, category, split="test")
    pairs = [(row["prompt_1"], row["prompt_2"]) for row in ds]
    print(f"   📂 {category}: {len(pairs)} prompt pairs loaded")
    return pairs


print("✅ Serialization utilities defined.")

✅ Serialization utilities defined.


# CELL 6.

In [10]:
from optipfair.bias import analyze_neuron_bias, compute_fairness_pruning_scores

all_bias_scores     = {}
all_fairness_scores = {}

for category in CATEGORIES:
    out_dir = f"{RESULTS_BASE}/{category}"
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  {category}")
    print(f"{'='*60}")

    # Load prompt pairs
    pairs = load_prompt_pairs(DATASET_NAME, category)

    # Bias analysis
    print(f"\n→ analyze_neuron_bias (aggregation='{AGGREGATION}', batch_size={BATCH_SIZE}) ...")
    bias_scores = analyze_neuron_bias(
        model=model,
        tokenizer=tokenizer,
        prompt_pairs=pairs,
        target_layers=TARGET_LAYERS,
        aggregation=AGGREGATION,
        show_progress=True,
    )
    save_bias_scores(bias_scores, out_dir)

    # Fairness pruning scores
    print(f"\n→ compute_fairness_pruning_scores (bias_weight={BIAS_WEIGHT}) ...")
    fairness_scores = compute_fairness_pruning_scores(
        model=model,
        bias_scores=bias_scores,
        bias_weight=BIAS_WEIGHT,
    )
    save_fairness_scores(fairness_scores, out_dir)

    all_bias_scores[category]     = bias_scores
    all_fairness_scores[category] = fairness_scores

    print(f"\n✅ {category} complete.")

print(f"\n{'='*60}")
print(f"All categories processed: {CATEGORIES}")


  PhysicalAppearance
   📂 PhysicalAppearance: 15 prompt pairs loaded

→ analyze_neuron_bias (aggregation='mean', batch_size=4) ...


Analyzing bias across prompt pairs: 100%|██████████| 15/15 [00:01<00:00,  7.92it/s]


   💾 bias_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/PhysicalAppearance/bias_scores.pt
   💾 bias_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/PhysicalAppearance/bias_scores.json

→ compute_fairness_pruning_scores (bias_weight=0.45) ...
   💾 fairness_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/PhysicalAppearance/fairness_scores.pt
   💾 fairness_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/PhysicalAppearance/fairness_scores.json

✅ PhysicalAppearance complete.

  Gender


test.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/15 [00:00<?, ? examples/s]

   📂 Gender: 15 prompt pairs loaded

→ analyze_neuron_bias (aggregation='mean', batch_size=4) ...


Analyzing bias across prompt pairs: 100%|██████████| 15/15 [00:00<00:00, 15.35it/s]


   💾 bias_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/Gender/bias_scores.pt
   💾 bias_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/Gender/bias_scores.json

→ compute_fairness_pruning_scores (bias_weight=0.45) ...
   💾 fairness_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/Gender/fairness_scores.pt
   💾 fairness_scores → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/Gender/fairness_scores.json

✅ Gender complete.

All categories processed: ['PhysicalAppearance', 'Gender']


In [11]:
import math

# ── Configurable threshold ───────────────────────────────────────────────────
EXPLORE_TOP_PERCENT = 1.0

# ── Load fairness scores from disk (re-runnable independently) ───────────────
fs = {
    cat: torch.load(f"{RESULTS_BASE}/{cat}/fairness_scores.pt", map_location="cpu")
    for cat in CATEGORIES
}

# ── Global top-N% per category ───────────────────────────────────────────────
def top_neurons_global(fairness_scores: dict, top_percent: float) -> tuple:
    all_entries = [
        (score.item(), int(layer_idx), neuron_idx)
        for layer_idx, scores in fairness_scores.items()
        for neuron_idx, score in enumerate(scores)
    ]
    total = len(all_entries)
    k = max(1, math.ceil(total * top_percent / 100.0))
    all_entries.sort(key=lambda x: x[0], reverse=True)
    result = {}
    for _, layer_idx, neuron_idx in all_entries[:k]:
        result.setdefault(layer_idx, set()).add(neuron_idx)
    return result, k, total

top = {}
for cat in CATEGORIES:
    top[cat], k, total = top_neurons_global(fs[cat], EXPLORE_TOP_PERCENT)

# ── N-way intersection (neurons shared across ALL categories) ────────────────
all_layers = sorted(set(l for cat in CATEGORIES for l in top[cat]))

intersection = {}
for layer_idx in all_layers:
    sets = [top[cat].get(layer_idx, set()) for cat in CATEGORIES]
    shared = sorted(set.intersection(*sets))
    if shared:
        intersection[layer_idx] = shared

total_shared = sum(len(v) for v in intersection.values())

# ── Report ───────────────────────────────────────────────────────────────────
col_width = 8
header_cats = "".join(f"{cat[:6]:>{col_width}}" for cat in CATEGORIES)
print(f"Top {EXPLORE_TOP_PERCENT}% threshold (global ranking) — {len(CATEGORIES)} categories")
print(f"  N-way intersection (shared across ALL): {total_shared} neurons")
print()
print(f"  {'Layer':<8}{header_cats}{'Shared':>{col_width}}")
print(f"  {'-' * (8 + col_width * len(CATEGORIES) + col_width)}")

for layer_idx in all_layers:
    counts = "".join(f"{len(top[cat].get(layer_idx, set())):>{col_width}}" for cat in CATEGORIES)
    n_shared = len(intersection.get(layer_idx, []))
    marker = " ◀" if n_shared > 0 else ""
    print(f"  {layer_idx:<8}{counts}{n_shared:>{col_width}}{marker}")

# ── Save comparison_summary.json ─────────────────────────────────────────────
summary = {
    "config": {
        "model"        : MODEL_NAME,
        "dataset"      : DATASET_NAME,
        "bias_weight"  : BIAS_WEIGHT,
        "top_percent"  : EXPLORE_TOP_PERCENT,
        "target_layers": TARGET_LAYERS,
        "aggregation"  : AGGREGATION,
        "categories"   : CATEGORIES,
    },
    **{
        cat: {
            "top_neurons_by_layer": {str(k): sorted(v) for k, v in top[cat].items()},
            "total_candidates"    : sum(len(v) for v in top[cat].values()),
        }
        for cat in CATEGORIES
    },
    "intersection": {
        "neurons_by_layer"   : {str(k): v for k, v in intersection.items()},
        "total"              : total_shared,
        "layers_with_overlap": len(intersection),
    },
}

summary_path = f"{RESULTS_BASE}/comparison_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n💾 comparison_summary.json → {summary_path}")
print(f"✅ Cross-category analysis complete.")

Top 1.0% threshold (global ranking) — 2 categories
  N-way intersection (shared across ALL): 171 neurons

  Layer     Physic  Gender  Shared
  --------------------------------
  0            538     449      72 ◀
  1            466     720      97 ◀
  2             18      57       1 ◀
  3             25       4       0
  4              2       4       0
  5             17       8       0
  6              4       2       0
  7             14       9       0
  8             19       5       0
  9             15       2       0
  10            10       7       0
  11             4      10       0
  12            17       8       0
  13            87       9       0
  14            71       7       1 ◀
  15             4      10       0

💾 comparison_summary.json → /content/drive/MyDrive/fairness-pruning/results/llama-3.2-1B/bias_detection/en/comparison_summary.json
✅ Cross-category analysis complete.


In [12]:
# ── Configurable threshold for this exploration ─────────────────────────────
EXPLORE_TOP_PERCENT = .5   # ← edit freely, independent of TOP_PERCENT above

# ── Load fairness scores from disk ──────────────────────────────────────────
fs_pa  = torch.load(f"{RESULTS_BASE}/PhysicalAppearance/fairness_scores.pt", map_location="cpu")
fs_gen = torch.load(f"{RESULTS_BASE}/Gender/fairness_scores.pt",             map_location="cpu")

# ── Global top-N%: pool all neurons across all layers, rank globally ─────────
def top_neurons_global(fairness_scores: dict, top_percent: float) -> dict:
    # Build a flat list of (score, layer_idx, neuron_idx)
    all_entries = [
        (score.item(), int(layer_idx), neuron_idx)
        for layer_idx, scores in fairness_scores.items()
        for neuron_idx, score in enumerate(scores)
    ]
    total_neurons = len(all_entries)
    k = max(1, math.ceil(total_neurons * top_percent / 100.0))

    # Sort descending by score, take top-k globally
    all_entries.sort(key=lambda x: x[0], reverse=True)
    top_entries = all_entries[:k]

    # Re-group by layer
    result = {}
    for _, layer_idx, neuron_idx in top_entries:
        result.setdefault(layer_idx, set()).add(neuron_idx)
    return result, k, total_neurons

top_pa,  k_pa,  total_pa  = top_neurons_global(fs_pa,  EXPLORE_TOP_PERCENT)
top_gen, k_gen, total_gen = top_neurons_global(fs_gen, EXPLORE_TOP_PERCENT)

# ── Intersection ─────────────────────────────────────────────────────────────
shared_layers = sorted(set(top_pa.keys()) & set(top_gen.keys()))
intersection  = {l: sorted(top_pa[l] & top_gen[l]) for l in shared_layers if top_pa[l] & top_gen[l]}
total_shared  = sum(len(v) for v in intersection.values())

# ── Report ───────────────────────────────────────────────────────────────────
print(f"Top {EXPLORE_TOP_PERCENT}% threshold (global ranking)")
print(f"  Total neurons in model        : {total_pa}  (PA) / {total_gen}  (Gender)")
print(f"  PhysicalAppearance candidates : {k_pa}")
print(f"  Gender candidates             : {k_gen}")
print(f"  Shared (intersection)         : {total_shared}")
print()
print(f"  {'Layer':<8} {'PA':>6} {'Gender':>8} {'Shared':>8}")
print(f"  {'-'*34}")
for layer_idx in sorted(set(top_pa.keys()) | set(top_gen.keys())):
    n_pa     = len(top_pa.get(layer_idx, set()))
    n_gen    = len(top_gen.get(layer_idx, set()))
    n_shared = len(intersection.get(layer_idx, []))
    marker   = " ◀" if n_shared > 0 else ""
    print(f"  {layer_idx:<8} {n_pa:>6} {n_gen:>8} {n_shared:>8}{marker}")

Top 0.5% threshold (global ranking)
  Total neurons in model        : 131072  (PA) / 131072  (Gender)
  PhysicalAppearance candidates : 656
  Gender candidates             : 656
  Shared (intersection)         : 68

  Layer        PA   Gender   Shared
  ----------------------------------
  0           256      229       34 ◀
  1           233      328       32 ◀
  2            12       23        1 ◀
  3            16        3        0
  4             1        4        0
  5            13        8        0
  6             3        2        0
  7             9        8        0
  8            18        5        0
  9            10        1        0
  10            7        7        0
  11            4        8        0
  12           10        6        0
  13           40        8        0
  14           21        6        1 ◀
  15            3       10        0
